In [ ]:
# Build one gold table of sales KPIs from the silver layer.
# Grain: one row per product category and quarter.
sales_order_detail = spark.table("adventure_works.silver.sales_order_detail")
product = spark.table("adventure_works.silver.product")

lines = sales_order_detail.join(product, "ProductID")

In [ ]:
from pyspark.sql import functions as F

kpis = (
    lines
    # ProductLine mapped to readable names; NULL means Other, not a dropped row.
    .withColumn("category", F.when(F.col("ProductLine") == "R", "Road")
        .when(F.col("ProductLine") == "M", "Mountain")
        .when(F.col("ProductLine") == "S", "Standard")
        .when(F.col("ProductLine") == "T", "Touring")
        .otherwise("Other"))
    # No sales_order_header in this lakehouse, so ModifiedDate is the date proxy.
    .withColumn("quarter", F.concat(F.year("ModifiedDate"), F.lit("-Q"), F.quarter("ModifiedDate")))
    .groupBy("category", "quarter")
    .agg(
        F.sum("LineTotal").alias("total_revenue"),
        F.countDistinct("SalesOrderID").alias("order_count"),
    )
    .withColumn("avg_order_value", F.col("total_revenue") / F.col("order_count"))
)

kpis.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("adventure_works.gold.sales_kpis_by_category_quarter")
print("rows written:", kpis.count())

In [ ]:
# Top 10 rows by revenue
display(spark.sql('''
SELECT *
FROM adventure_works.gold.sales_kpis_by_category_quarter
ORDER BY total_revenue DESC
LIMIT 10
'''))

In [ ]:
# Sanity check: avg_order_value must equal total_revenue / order_count on every row
bad_rows = spark.sql('''
SELECT COUNT(*) AS n
FROM adventure_works.gold.sales_kpis_by_category_quarter
WHERE avg_order_value <> total_revenue / order_count
''').collect()[0][0]
print("rows where avg_order_value does not check out:", bad_rows)